# 02. Gaussian Naive Bayes: Variables Continuas, Geometría de Fronteras y Transformaciones

**Módulo:** `05_Naive_Bayes`  
**Objetivo:** Dominar la formulación matemática de *Gaussian Naive Bayes* (GNB) para espacios continuos $\mathbb{R}^D$, deducir analíticamente los estimadores de Máxima Verosimilitud (MLE), comprender por qué sus fronteras de decisión son secciones cónicas (cuadráticas/elípticas), analizar el efecto estabilizador de `var_smoothing`, corregir violaciones al supuesto de normalidad mediante transformaciones de potencia (`PowerTransformer`) y demostrar la invarianza del modelo frente al escalamiento lineal.

---

## 1. Formulación Matemática de Gaussian Naive Bayes

Cuando las características de entrada son continuas, $\mathbf{x} = (x_1, x_2, \dots, x_D) \in \mathbb{R}^D$, ya no podemos estimar tablas de frecuencias discretas sin recurrir a la discretización (*binning*), lo cual introduce pérdida de información y sensibilidad a los cortes. En su lugar, **Gaussian Naive Bayes** asume que, condicionada a la clase $Y = c_k$, cada característica continua $X_j$ sigue una **distribución normal (Gaussiana) univariada**:

$$X_j \mid (Y = c_k) \sim \mathcal{N}\left(\mu_{kj}, \sigma_{kj}^2\right)$$

La función de densidad de probabilidad (*PDF*) para la $j$-ésima característica en la clase $c_k$ está dada por:

$$p(x_j \mid Y = c_k) = \frac{1}{\sqrt{2\pi\sigma_{kj}^2}} \exp\left( -\frac{(x_j - \mu_{kj})^2}{2\sigma_{kj}^2} \right)$$

### Perspectiva Multivariada: Matriz de Covarianza Diagonal
Bajo el supuesto "ingenuo" de independencia condicional ($X_i \perp\!\!\!\perp X_j \mid Y$), la densidad conjunta condicional es el producto de las $D$ densidades univariadas. Esto equivale exactamente a modelar cada clase mediante una **Distribución Normal Multivariada** $\mathcal{N}(\boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$ cuya matriz de covarianza $\boldsymbol{\Sigma}_k$ está restringida a ser **estrictamente diagonal**:

$$\boldsymbol{\Sigma}_k = \text{diag}\left(\sigma_{k1}^2, \sigma_{k2}^2, \dots, \sigma_{kD}^2\right) = \begin{pmatrix} \sigma_{k1}^2 & 0 & \cdots & 0 \\ 0 & \sigma_{k2}^2 & \cdots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \cdots & \sigma_{kD}^2 \end{pmatrix}$$

Geométricamente, esto significa que los contornos de isodensidad de cada clase son **hiperelipsoides cuyos ejes principales están obligatoriamente alineados con los ejes coordenados** del espacio de características.

### Regla de Decisión MAP en Espacio Logarítmico
Sustituyendo la densidad Gaussiana en la función discriminante logarítmica $\ell_k(\mathbf{x}) = \ln P(Y = c_k) + \sum_{j=1}^{D} \ln p(x_j \mid Y = c_k)$, obtenemos la expresión exacta que evalúa Scikit-Learn internamente (`_joint_log_likelihood`):

$$\ell_k(\mathbf{x}) = \ln P(Y = c_k) - \frac{1}{2} \sum_{j=1}^{D} \ln\left(2\pi\sigma_{kj}^2\right) - \frac{1}{2} \sum_{j=1}^{D} \frac{(x_j - \mu_{kj})^2}{\sigma_{kj}^2}$$

In [ ]:
# Configuración del entorno y librerías estándar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.datasets import load_wine, make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import accuracy_score, f1_score, classification_report
from IPython.display import display


# Estándar visual del repositorio
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)

print("Entorno configurado correctamente.")

---

## 2. Estimación de Parámetros por Máxima Verosimilitud (MLE)

¿De dónde salen los valores $\mu_{kj}$ y $\sigma_{kj}^2$ durante el método `.fit(X, y)`? Se obtienen maximizando la función de log-verosimilitud sobre el subconjunto de muestras $\mathcal{D}_k$ que pertenecen a la clase $c_k$ (con cardinalidad $N_k = |\mathcal{D}_k|$).

Para una característica fija $j$ y clase $c_k$, la función de log-verosimilitud de los datos observados $\{x_{1j}, x_{2j}, \dots, x_{N_k j}\}$ es:

$$\mathcal{L}(\mu_{kj}, \sigma_{kj}^2) = \sum_{i \in \mathcal{D}_k} \ln p(x_{ij} \mid \mu_{kj}, \sigma_{kj}^2) = -\frac{N_k}{2} \ln(2\pi) - \frac{N_k}{2} \ln(\sigma_{kj}^2) - \frac{1}{2\sigma_{kj}^2} \sum_{i \in \mathcal{D}_k} (x_{ij} - \mu_{kj})^2$$

### Paso 1: Derivada parcial respecto a la media $\mu_{kj}$
Derivamos $\mathcal{L}$ respecto a $\mu_{kj}$ e igualamos a cero:

$$\frac{\partial \mathcal{L}}{\partial \mu_{kj}} = \frac{1}{\sigma_{kj}^2} \sum_{i \in \mathcal{D}_k} (x_{ij} - \mu_{kj}) = 0 \implies \sum_{i \in \mathcal{D}_k} x_{ij} - N_k \mu_{kj} = 0$$

$$\hat{\mu}_{kj}^{\text{MLE}} = \frac{1}{N_k} \sum_{i \in \mathcal{D}_k} x_{ij}$$

### Paso 2: Derivada parcial respecto a la varianza $\sigma_{kj}^2$
Sea $v = \sigma_{kj}^2$. Derivamos $\mathcal{L}$ respecto a $v$ e igualamos a cero:

$$\frac{\partial \mathcal{L}}{\partial v} = -\frac{N_k}{2v} + \frac{1}{2v^2} \sum_{i \in \mathcal{D}_k} (x_{ij} - \hat{\mu}_{kj})^2 = 0$$

Multiplicando ambos lados por $2v^2 / N_k$, obtenemos el estimador de máxima verosimilitud para la varianza:

$$\hat{\sigma}_{kj}^{2 \, (\text{MLE})} = \frac{1}{N_k} \sum_{i \in \mathcal{D}_k} \left(x_{ij} - \hat{\mu}_{kj}\right)^2$$

> **Nota Técnica Crítica en Producción:** Observa que el denominador del estimador MLE es **$N_k$** (`ddof=0` en NumPy) y **no** $N_k - 1$ (corrección de Bessel para la varianza muestral insesgada). Scikit-Learn utiliza estrictamente el estimador MLE ($1/N_k$) más un término de regularización numérica `var_smoothing`.

--- 

## 3. Inspección Profunda de Atributos Internos en Scikit-Learn

Al entrenar una instancia de `GaussianNB`, Scikit-Learn almacena los parámetros aprendidos en atributos terminados con guion bajo (`_`):
* `classes_`: Arreglo `(C,)` con las etiquetas únicas de clase.
* `class_count_`: Arreglo `(C,)` con el número de muestras $N_k$ por clase.
* `class_prior_`: Arreglo `(C,)` con las probabilidades a priori $P(Y = c_k) = N_k / N$.
* `theta_`: Matriz `(C, D)` que contiene las medias estimadas $\hat{\mu}_{kj}$.
* `var_`: Matriz `(C, D)` que contiene las varianzas estimadas $\hat{\sigma}_{kj}^2 + \epsilon$.

In [ ]:
# Carga del dataset Wine y verificación empírica de los atributos MLE en Scikit-Learn
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
feature_names = wine.feature_names

# Entrenamos con var_smoothing=0 para verificar la fórmula MLE exacta primero
gnb_exact = GaussianNB(var_smoothing=0.0).fit(X_wine, y_wine)

# Cálculo manual en NumPy para la Clase 0
X_c0 = X_wine[y_wine == 0]
mu_c0_manual = np.mean(X_c0, axis=0)
var_c0_manual = np.var(X_c0, axis=0, ddof=0) # MLE usa ddof=0 (divide entre N_k)

print("=== AUDITORÍA DE ATRIBUTOS INTERNOS DE GAUSSIANNB ===")
print(f"Clases detectadas (classes_):       {gnb_exact.classes_}")
print(f"Conteo por clase (class_count_):    {gnb_exact.class_count_}")
print(f"Priors por clase (class_prior_):    {np.round(gnb_exact.class_prior_, 4)}")
print(f"Dimensiones de theta_ (Medias):     {gnb_exact.theta_.shape}")
print(f"Dimensiones de var_ (Varianzas):    {gnb_exact.var_.shape}\n")

diff_theta = np.max(np.abs(gnb_exact.theta_[0] - mu_c0_manual))
diff_var = np.max(np.abs(gnb_exact.var_[0] - var_c0_manual))

print(f"Diferencia máxima en Media (theta_[0] vs np.mean):        {diff_theta:.2e}")
print(f"Diferencia máxima en Varianza (var_[0] vs np.var ddof=0): {diff_var:.2e}")

# Visualización de las campanas Gaussianas aprendidas para las 2 primeras características
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
colores = ['#1f77b4', '#ff7f0e', '#2ca02c']

for j, ax in enumerate(axes):
    x_min, x_max = X_wine[:, j].min() - 1.0, X_wine[:, j].max() + 1.0
    x_Axis = np.linspace(x_min, x_max, 300)
    
    for c_idx, c_name in enumerate(wine.target_names):
        mu = gnb_exact.theta_[c_idx, j]
        sigma = np.sqrt(gnb_exact.var_[c_idx, j])
        pdf = stats.norm.pdf(x_Axis, loc=mu, scale=sigma)
        
        ax.plot(x_Axis, pdf, color=colores[c_idx], lw=2.5, label=f"{c_name} ($\mu={mu:.2f}, \sigma={sigma:.2f}$)")
        ax.fill_between(x_Axis, pdf, color=colores[c_idx], alpha=0.2)
        
    ax.set_title(f"Densidades Aprendidas $p(x_j \mid Y)$: {feature_names[j]}", fontsize=12, fontweight='bold')
    ax.set_xlabel(feature_names[j], fontsize=11)
    ax.set_ylabel("Densidad $p(x_j \mid Y=c_k)$", fontsize=11)
    ax.legend(frameon=True, facecolor='white', fontsize=9.5)

plt.tight_layout()
plt.show()

---

## 4. Geometría de las Fronteras de Decisión en 2D

Un mito común entre principiantes es pensar que Naive Bayes es siempre un clasificador lineal. Analicemos matemáticamente la frontera de decisión entre dos clases $c_1$ y $c_2$ en $\mathbb{R}^D$. La frontera ocurre donde los puntajes logarítmicos se igualan: $\ell_1(\mathbf{x}) = \ell_2(\mathbf{x})$:

$$\ln \frac{P(Y=c_1)}{P(Y=c_2)} - \frac{1}{2}\sum_{j=1}^{D} \ln \frac{\sigma_{1j}^2}{\sigma_{2j}^2} - \frac{1}{2} \sum_{j=1}^{D} \left[ \frac{(x_j - \mu_{1j})^2}{\sigma_{1j}^2} - \frac{(x_j - \mu_{2j})^2}{\sigma_{2j}^2} \right] = 0$$

Al expandir los cuadrados $(x_j - \mu_{kj})^2 = x_j^2 - 2\mu_{kj}x_j + \mu_{kj}^2$, podemos agrupar la ecuación como un polinomio de segundo grado en las coordenadas $x_j$:

$$\sum_{j=1}^{D} a_j x_j^2 + \sum_{j=1}^{D} b_j x_j + c = 0 \quad \text{donde} \quad a_j = -\frac{1}{2}\left(\frac{1}{\sigma_{1j}^2} - \frac{1}{\sigma_{2j}^2}\right)$$

### Conclusiones Geométricas Fundamentales:
1. **Varianzas Heterogéneas ($\sigma_{1j}^2 \neq \sigma_{2j}^2$) $\implies$ Frontera Cuadrática:**  
   Como en los datos reales cada clase tiene su propia dispersión ($a_j \neq 0$), la ecuación define una **sección cónica** en 2D: **parábolas, elipses o hipérbolas** (sin términos cruzados $x_1 x_2$ porque asumimos independencia condicional).
2. **Varianzas Homogéneas ($\sigma_{1j}^2 = \sigma_{2j}^2$) $\implies$ Frontera Lineal:**  
   Únicamente cuando las varianzas son idénticas entre clases, el coeficiente cuadrático se anula ($a_j = 0$) y la frontera colapsa a un hiperplano lineal (caso particular equivalente a LDA con covarianza diagonal).
3. **Diferencia frente a QDA (*Quadratic Discriminant Analysis*):**  
   QDA estima la matriz de covarianza completa $\boldsymbol{\Sigma}_k$ (incluyendo covarianzas fuera de la diagonal $x_1 x_2$), lo que le permite rotar las elipses en cualquier ángulo, a costa de estimar $\mathcal{O}(C \cdot D^2)$ parámetros en lugar de $\mathcal{O}(C \cdot D)$.

In [ ]:
# Comparación Geométrica 2D: LDA (Lineal) vs GaussianNB (Cuadrática Alineada) vs QDA (Cuadrática Rotada)
np.random.seed(42)

# Generamos un dataset 2D donde las clases tienen varianzas muy distintas y cierta rotación
N_pts = 250
X_class0 = np.random.multivariate_normal(mean=[0.0, 0.0], cov=[[0.4, 0.1], [0.1, 0.4]], size=N_pts)
X_class1 = np.random.multivariate_normal(mean=[1.8, 1.5], cov=[[3.2, -1.8], [-1.8, 2.5]], size=N_pts)

X_2d = np.vstack([X_class0, X_class1])
y_2d = np.array([0]*N_pts + [1]*N_pts)

modelos_2d = [
    ("LDA (Covarianza Compartida -> Lineal)", LinearDiscriminantAnalysis()),
    ("GaussianNB (Varianza Diagonal -> Elipse/Parábola)", GaussianNB()),
    ("QDA (Covarianza Completa -> Cónica Rotada)", QuadraticDiscriminantAnalysis())
]

fig, axes = plt.subplots(1, 3, figsize=(17, 5.2), sharex=True, sharey=True)

for ax, (titulo, clf) in zip(axes, modelos_2d):
    clf.fit(X_2d, y_2d)
    acc = clf.score(X_2d, y_2d)
    
    # Superficie de probabilidad y frontera con DecisionBoundaryDisplay
    DecisionBoundaryDisplay.from_estimator(
        clf, X_2d, response_method="predict_proba", cmap="RdBu_r", alpha=0.45, ax=ax, eps=0.8
    )
    DecisionBoundaryDisplay.from_estimator(
        clf, X_2d, response_method="predict", plot_method="contour", colors="black", linewidths=2.0, ax=ax, eps=0.8
    )
    
    ax.scatter(X_class0[:, 0], X_class0[:, 1], c='#1f77b4', edgecolors='k', s=30, alpha=0.8, label='Clase 0 (Compacta)')
    ax.scatter(X_class1[:, 0], X_class1[:, 1], c='#d62728', edgecolors='k', s=30, alpha=0.8, label='Clase 1 (Dispersa)')
    ax.set_title(f"{titulo}\nAccuracy: {acc:.2%}", fontsize=11.5, fontweight='bold')
    ax.set_xlabel("Característica $X_1$", fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel("Característica $X_2$", fontsize=11)
    ax.legend(loc='lower left', frameon=True, facecolor='white', fontsize=9)

plt.suptitle("Geometría de las Fronteras de Decisión Generativas en 2D", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 5. El Hiperparámetro `var_smoothing`: Estabilidad Numérica y Regularización

### El Peligro de las Varianzas Nulas o Degeneradas
Observa nuevamente la fórmula de log-verosimilitud de GaussianNB:

$$\ell_k(\mathbf{x}) = \ln P(Y = c_k) - \frac{1}{2} \sum_{j=1}^{D} \ln\left(2\pi\sigma_{kj}^2\right) - \frac{1}{2} \sum_{j=1}^{D} \frac{(x_j - \mu_{kj})^2}{\sigma_{kj}^2}$$

¿Qué ocurre si una característica $X_j$ es constante dentro de los datos de entrenamiento de la clase $c_k$ (por ejemplo, un sensor digital que vale `0.0` en todas las muestras de una clase minoritaria)?
* La varianza muestral es $\hat{\sigma}_{kj}^2 = 0$.
* Al evaluar cualquier muestra en producción donde $x_j \neq \mu_{kj}$, obtenemos una **división por cero** ($(x_j - \mu_{kj})^2 / 0 = \infty$) y $\ln(0) = -\infty$, destruyendo el cálculo numérico con valores `NaN` o `-inf`.

### Solución Matemática: `var_smoothing`
Scikit-Learn resuelve esto sumando una pequeña fracción $\varepsilon$ (`var_smoothing`, por defecto $10^{-9}$) de la **mayor varianza encontrada entre todas las características del dataset** a todas las varianzas estimadas:

$$\tilde{\sigma}_{kj}^2 = \hat{\sigma}_{kj}^2 + \varepsilon \cdot \max_{m=1,\dots,D} \text{Var}(X_m)$$

### Rol Dual:
1. **Para $\varepsilon = 10^{-9}$ (Valor por defecto):** Actúa puramente como un salvavidas numérico contra divisiones por cero.
2. **Para valores mayores ($\varepsilon \in [10^{-3}, 1.0]$):** Actúa como un **hiperparámetro de regularización (Shrinkage)**. Al incrementar $\varepsilon$, ensancha artificialmente las campanas Gaussianas, haciendo que las varianzas entre clases se vuelvan más homogéneas y **suavizando fronteras cuadráticas sobreajustadas hacia fronteras más lineales y estables**.

In [ ]:
# Visualización del Efecto Regularizador de var_smoothing sobre las Fronteras en el Dataset Wine
X_wine_2d = X_wine[:, [0, 9]] # Alcohol vs Color Intensity
smoothing_values = [1e-9, 1e-1, 1.5]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharex=True, sharey=True)
palette_wine = ['#1f77b4', '#ff7f0e', '#2ca02c']

for ax, eps_val in zip(axes, smoothing_values):
    clf_smooth = GaussianNB(var_smoothing=eps_val).fit(X_wine_2d, y_wine)
    acc_val = clf_smooth.score(X_wine_2d, y_wine)
    
    DecisionBoundaryDisplay.from_estimator(
        clf_smooth, X_wine_2d, response_method="predict", cmap="Pastel1", alpha=0.65, ax=ax, eps=0.5
    )
    DecisionBoundaryDisplay.from_estimator(
        clf_smooth, X_wine_2d, response_method="predict", plot_method="contour", colors="k", linewidths=1.2, alpha=0.7, ax=ax, eps=0.5
    )
    
    for c_idx, c_name in enumerate(wine.target_names):
        mask = (y_wine == c_idx)
        ax.scatter(X_wine_2d[mask, 0], X_wine_2d[mask, 1], c=palette_wine[c_idx],
                   edgecolors='k', s=35, label=c_name)
        
    ax.set_title(f"var_smoothing = {eps_val:.0e}\nAccuracy: {acc_val:.2%}", fontsize=11.5, fontweight='bold')
    ax.set_xlabel("Alcohol", fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel("Color Intensity", fontsize=11)
    ax.legend(loc='upper left', frameon=True, facecolor='white', fontsize=9)

plt.suptitle("Efecto de Regularización Geométrica del Hiperparámetro var_smoothing", fontsize=13.5, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 6. El Supuesto de Normalidad y Transformaciones No Lineales (`PowerTransformer`)

### ¿Qué ocurre cuando los datos tienen asimetría severa (*Skewness*)?
En problemas industriales reales (ingresos financieros, tiempos de latencia de red, concentraciones químicas, conteos de actividad), las variables continuas rara vez son simétricas; suelen presentar **colas pesadas a la derecha** (distribuciones Log-Normal, Pareto o Gamma).

Cuando ajustamos una campana Gaussiana simétrica sobre una distribución fuertemente asimétrica:
1. La media $\hat{\mu}_{kj}$ y la varianza $\hat{\sigma}_{kj}^2$ se ven arrastradas por los valores extremos de la cola derecha.
2. El modelo asigna una densidad de probabilidad irrealmente alta a valores negativos o inexistentes en la cola izquierda, degradando severamente la frontera de decisión.

### Solución en Ingeniería de Características: Transformación de Yeo-Johnson
La familia de transformaciones paramétricas monotónicas de **Box-Cox** (solo para $x > 0$) y **Yeo-Johnson** (admite cualquier $x \in \mathbb{R}$, incluyendo ceros y negativos) busca encontrar un exponente óptimo $\lambda_j$ por máxima verosimilitud para cada característica con el fin de estabilizar la varianza y acercar su distribución a una Gaussiana estándar $\mathcal{N}(0, 1)$:

$$\psi(\lambda, x) = \begin{cases} \frac{(x + 1)^\lambda - 1}{\lambda} & \text{si } \lambda \neq 0, \; x \geq 0 \\ \ln(x + 1) & \text{si } \lambda = 0, \; x \geq 0 \\ -\frac{(-x + 1)^{2 - \lambda} - 1}{2 - \lambda} & \text{si } \lambda \neq 2, \; x < 0 \\ -\ln(-x + 1) & \text{si } \lambda = 2, \; x < 0 \end{cases}$$

In [ ]:
# Diagnóstico de Asimetría (Skewness) y Mejora de GaussianNB con PowerTransformer
np.random.seed(42)

# Generamos un dataset multiclase realista y aplicamos una transformación exponencial
# para simular características financieras/químicas con fuerte sesgo positivo (Log-Normal)
X_base, y_skew = make_classification(
    n_samples=1200, n_features=10, n_informative=8, n_redundant=2,
    n_classes=3, class_sep=1.1, random_state=42
)
# Introducimos asimetría severa en las primeras 6 variables
X_skewed = X_base.copy()
for j in range(6):
    X_skewed[:, j] = np.exp(0.85 * X_base[:, j])

# Calculamos el coeficiente de asimetría de Fisher-Pearson con SciPy
skew_antes = stats.skew(X_skewed[:, 0])
pt = PowerTransformer(method='yeo-johnson', standardize=True)
X_transformed_sample = pt.fit_transform(X_skewed)
skew_despues = stats.skew(X_transformed_sample[:, 0])

# Visualización del efecto de PowerTransformer (Histograma + Q-Q Plot)
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Antes: Histograma
sns.histplot(X_skewed[:, 0], kde=True, ax=axes[0, 0], color='#d62728', bins=40)
axes[0, 0].set_title(f"Característica $X_0$ Original (Asimetría = {skew_antes:.2f})", fontweight='bold')

# Antes: Q-Q Plot
stats.probplot(X_skewed[:, 0], dist="norm", plot=axes[0, 1])
axes[0, 1].set_title("Q-Q Plot Original (Severa Desviación de Normalidad)", fontweight='bold')

# Después: Histograma
sns.histplot(X_transformed_sample[:, 0], kde=True, ax=axes[1, 0], color='#2ca02c', bins=40)
axes[1, 0].set_title(f"Característica $X_0$ tras Yeo-Johnson (Asimetría = {skew_despues:.2f})", fontweight='bold')

# Después: Q-Q Plot
stats.probplot(X_transformed_sample[:, 0], dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("Q-Q Plot tras Yeo-Johnson (Alineación Gaussiana)", fontweight='bold')

plt.tight_layout()
plt.show()

# Evaluación rigurosa con Validación Cruzada Estratificada (10-Fold)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

pipe_crudo = Pipeline([
    ('gnb', GaussianNB())
])

pipe_yeo_johnson = Pipeline([
    ('power', PowerTransformer(method='yeo-johnson', standardize=True)),
    ('gnb', GaussianNB())
])

scores_crudo = cross_val_score(pipe_crudo, X_skewed, y_skew, cv=cv, scoring='accuracy')
scores_pt = cross_val_score(pipe_yeo_johnson, X_skewed, y_skew, cv=cv, scoring='accuracy')

print("=== IMPACTO DE POWERTRANSFORMER EN EL RENDIMIENTO DE GAUSSIANNB (10-Fold CV) ===")
print(f"Exactitud sin transformar (Datos Asimétricos): {scores_crudo.mean():.2%} ± {scores_crudo.std():.2%}")
print(f"Exactitud con Pipeline PowerTransformer:       {scores_pt.mean():.2%} ± {scores_pt.std():.2%}")
print(f"Ganancia neta de exactitud:                    +{(scores_pt.mean() - scores_crudo.mean())*100:.2f} puntos porcentuales")

---

## 7. Invarianza de GaussianNB a la Escala Lineal de las Características

Una pregunta frecuente en entrevistas técnicas de Ciencia de Datos es:
> *"¿Es necesario aplicar `StandardScaler` o `MinMaxScaler` antes de entrenar un `GaussianNB`?"*

### Demostración Analítica de Invarianza Afín
Supongamos que aplicamos una transformación lineal afín a cada característica $X_j$: sea $Z_j = a_j X_j + b_j$ con $a_j > 0$.
1. Por propiedades de la esperanza y la varianza:
   $$\mu_{Z, kj} = a_j \mu_{X, kj} + b_j \quad \text{y} \quad \sigma_{Z, kj}^2 = a_j^2 \sigma_{X, kj}^2$$
2. El término cuadrático estandarizado (distancia de Mahalanobis univariada) permanece **exactamente invariante** porque los factores de escala $a_j$ y traslación $b_j$ se cancelan algebraicamente:
   $$\frac{(z_j - \mu_{Z, kj})^2}{\sigma_{Z, kj}^2} = \frac{(a_j x_j + b_j - (a_j \mu_{X, kj} + b_j))^2}{a_j^2 \sigma_{X, kj}^2} = \frac{a_j^2 (x_j - \mu_{X, kj})^2}{a_j^2 \sigma_{X, kj}^2} = \frac{(x_j - \mu_{X, kj})^2}{\sigma_{X, kj}^2}$$
3. En cuanto al término logarítmico de normalización:
   $$-\frac{1}{2}\sum_{j=1}^{D} \ln\left(2\pi a_j^2 \sigma_{X, kj}^2\right) = -\frac{1}{2}\sum_{j=1}^{D} \ln\left(2\pi \sigma_{X, kj}^2\right) - \underbrace{\sum_{j=1}^{D} \ln(a_j)}_{\text{Constante } \forall c_k}$$
   Dado que la suma $-\sum_{j=1}^D \ln(a_j)$ **no depende de la clase $c_k$**, suma la misma constante al puntaje logarítmico de todas las clases y se cancela por completo al aplicar la normalización *Log-Sum-Exp*.

**Matiz de Ingeniería en Scikit-Learn (`var_smoothing`):**  
En la teoría exacta ($\varepsilon = 0$), `StandardScaler` no altera ni un solo bit las probabilidades posteriores. En la práctica con Scikit-Learn ($\varepsilon > 0$), dado que `var_smoothing` multiplica a $\max_m \text{Var}(X_m)$, si una variable está medida en millones (ej. salario anual en pesos) y otra en milésimas, $\max_m \text{Var}(X_m)$ será gigantesco y sobre-suavizará las variables de escala pequeña. Por tanto, **cuando `var_smoothing` es grande o las escalas difieren por muchos órdenes de magnitud, `StandardScaler` sí estabiliza el efecto uniforme de `var_smoothing`**.

In [ ]:
# Verificación Empírica de la Invarianza al Escalamiento Lineal (StandardScaler)
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.3, stratify=y_wine, random_state=42
)

# Caso 1: Sin var_smoothing (Teoría Gaussiana Pura)
gnb_raw_pure = GaussianNB(var_smoothing=0.0).fit(X_train_w, y_train_w)
pipe_scaled_pure = Pipeline([
    ('scaler', StandardScaler()),
    ('gnb', GaussianNB(var_smoothing=0.0))
]).fit(X_train_w, y_train_w)

prob_raw_pure = gnb_raw_pure.predict_proba(X_test_w)
prob_scaled_pure = pipe_scaled_pure.predict_proba(X_test_w)

diff_pure = np.max(np.abs(prob_raw_pure - prob_scaled_pure))
print(f"1. Diferencia máxima de probabilidades (Sin vs Con StandardScaler, var_smoothing=0):    {diff_pure:.2e}")

# Caso 2: Con var_smoothing por defecto (1e-9)
gnb_raw_def = GaussianNB(var_smoothing=1e-9).fit(X_train_w, y_train_w)
pipe_scaled_def = Pipeline([
    ('scaler', StandardScaler()),
    ('gnb', GaussianNB(var_smoothing=1e-9))
]).fit(X_train_w, y_train_w)

acc_raw_def = gnb_raw_def.score(X_test_w, y_test_w)
acc_scaled_def = pipe_scaled_def.score(X_test_w, y_test_w)
print(f"2. Exactitud en Test con var_smoothing=1e-9 -> Crudo: {acc_raw_def:.2%} | Escalado: {acc_scaled_def:.2%}")

---

## 8. Conclusiones y Guía Práctica para Producción

1. **Fronteras Cuadráticas Rápidas:** `GaussianNB` ofrece fronteras de decisión curvas (elípticas/parabólicas) con costo computacional lineal $\mathcal{O}(N \cdot D)$, sirviendo como un excelente *baseline* no lineal antes de recurrir a modelos pesados.
2. **Diagnóstico de Asimetría Obligatorio:** Antes de alimentar variables continuas a `GaussianNB`, inspecciona su asimetría (`scipy.stats.skew`). Si $|\text{skew}| > 1.0$, encapsula el modelo en un `Pipeline` con `PowerTransformer(method='yeo-johnson')`.
3. **Ajuste de `var_smoothing`:** Si tus características tienen escalas muy dispares (ej. `Proline` en miles vs `Nonflavanoid phenols` en décimas dentro del dataset Wine) y decides optimizar `var_smoothing` mediante búsqueda en malla (`GridSearchCV`), incluye siempre un `StandardScaler` previo para que el término $\varepsilon \cdot \max \text{Var}(X)$ actúe de manera isotrópica sobre todas las dimensiones.

---
**Siguiente paso:** En el **Notebook 03 (`03_Modelos_Discretos_NLP_y_Categoricos.ipynb`)** nos adentraremos en el modelado de conteos discretos, matrices dispersas de alta dimensionalidad, clasificación de texto (NLP) y manejo de desbalance severo con `MultinomialNB`, `BernoulliNB`, `CategoricalNB` y `ComplementNB`.